# TAE-IA · Module 6 · L24 — «¿Qué suena aquí?» · **the audio half**

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Session** | L24 — Track B, closing |
| **What this is** | The skeleton of an app. You write the body. |
| **Due** | Today in class, or as homework with no penalty |
| **Graded** | No. What counts is that it runs and that you show it |

---

## What you are building

An app that **listens to a clip and says what is in it**, using a vocabulary of labels
**you write yourself**.

```
        audio clip
             |
             v
    [ coerce_audio ]  ← the contract: float32 mono at the right rate
             |
      +------+------+
      |             |
      v             v
   [ CLAP ]     [ Whisper ]
    labels      any speech?
      |             |
      +------+------+
             v
      a report in Spanish
             |
             v
        [ Gradio ]
```

On Monday, in L25, **the other half**: the same app takes an image, tags it against
**the same vocabulary**, and compares. Does what you see match what you hear?

> That is why today's contract is not negotiable. Monday's image half plugs into exactly
> the shape today's functions return.

---

## The spec

Your app must, **at minimum**:

1. Accept an audio clip from Gradio and survive anything it is given.
2. Tag it with **CLAP** against a vocabulary you wrote.
3. Use **Whisper** to detect speech, and transcribe it when there is any.
4. Return a report in Spanish with: the top label, its confidence, the top-3, and the
   transcript when it applies.
5. **Refuse to guess** when confidence falls below a floor you set.
6. Run behind a Gradio interface with `gr.Audio` as its input.

And it must **pass the test cell** in section 6. That cell does not get modified.

## The models

| For | Model | Have you used it? |
|---|---|---|
| Tagging sound | `laion/clap-htsat-unfused` | **No. It is new.** Go to HuggingFace and read its card. |
| Detecting speech | Whisper `small` | Yes — L19, L20 |

CLAP does **zero-shot classification**: it has no fixed 50 classes like your L17 model. You hand
it a list of phrases and it says which one fits best. The vocabulary is **yours**, and so are its
mistakes.

## Rules

- **Use whatever you like**: your old notebooks, the documentation, an AI assistant.
- What you cannot do is hand in something that does not run. The test cell decides, not effort.
- If you do not finish today, take what you have and finish it as homework. No penalty.
- **Everyone shows something at the end of L25**, finished or not.

---

## 1 — Install and setup · **given, do not touch**

In [ ]:
# Two packages. CLAP rides inside `transformers`, which Colab already has.
!pip install -q openai-whisper librosa

In [ ]:
import os, sys, gc, time, random
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L24_output'
INPUT_DIR  = f'{DRIVE_ROOT}/inputs'          # the shared folder from L11/L12
for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

# Models live on the runtime disk, not on Drive. Wiped when the runtime recycles
# (~2 min to re-download), and in exchange there is no OSError 95 from symlinks.
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']        = MODEL_CACHE
os.environ['TORCH_HOME']     = MODEL_CACHE
os.environ['XDG_CACHE_HOME'] = MODEL_CACHE

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

## 2 — The models · **you load them**

Two models, and together they use under 2 GB of the T4's 15. Nothing to load and free:
both stay resident all day.

**CLAP.** Loads through the `transformers` `pipeline`. The task is called
`zero-shot-audio-classification`. The repo is `laion/clap-htsat-unfused`.

> ⚠️ This repo has **no `.safetensors`** — only `pytorch_model.bin`. If you copy a
> `snapshot_download(..., allow_patterns=['*.safetensors'])` from another notebook, nothing
> downloads and the error surfaces much later.

**Whisper.** Same as L19: `whisper.load_model('small', download_root=MODEL_CACHE)`.

In [ ]:
from transformers import pipeline
import whisper

# TODO: load CLAP into `tagger`.
#   pipeline('zero-shot-audio-classification', model=..., device=0)
tagger = None

vram('after CLAP')        # expect ~0.6 GB

# TODO: load Whisper small into `asr`.
asr = None

vram('both loaded')       # expect ~1.6 GB

## 3 — The contract · **given, and not modified**

This cell is half of the contract Monday's session closes. It is the only code you are handed,
and it is handed to you for one reason: **this is where the failure that does not crash lives.**

Three measured facts about CLAP, not opinions:

| Give it… | What happens |
|---|---|
| `int16` exactly as Gradio delivers it | Classifies rain as **"a dog barking", 0.576**. No exception. |
| The wrong sample rate | **Sometimes** right. Dog breaks at 22 kHz and 16 kHz; rain, birds, vacuum and clock all survive. |
| A `dict` with `sampling_rate` | `TypeError`. CLAP **cannot** know your rate — resampling is your job. |

The second is the dangerous one: five clips out of six forgive the bug, so your own test passes
and the person next to you gets nonsense.

In [ ]:
import librosa

# The rate each stage wants. Two different numbers in one pipeline.
CLAP_SR     = 48000     # measured: at any other rate CLAP fails intermittently
ASR_SR      = 16000     # Whisper
MAX_SECONDS = 60        # a 30-minute upload is the #1 cause of a stalled demo
MIN_SECONDS = 0.5


def list_inputs():
    """What is in the shared folder right now."""
    names = sorted(f for f in os.listdir(INPUT_DIR) if not f.startswith('.'))
    print(f'{INPUT_DIR}  ({len(names)} files)')
    for n in names:
        print(f'  {os.path.getsize(os.path.join(INPUT_DIR, n))/1e6:6.2f} MB  {n}')
    return names


def upload_inputs():
    """Pick files from your machine; they land in Drive and stay there."""
    from google.colab import files
    for fname, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
            f.write(data)
        print(f'  saved {fname}')


def load_audio(name_or_path, sr=None):
    """A name in the inputs folder, or any path -> (wav float32 mono, sr)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and pick it, or copy it into that folder on Drive.')
    wav, got_sr = librosa.load(path, sr=sr, mono=True)
    return wav.astype('float32'), got_sr


def coerce_audio(wav, sr, target_sr):
    """Anything a user can hand us -> float32 mono at target_sr, or ValueError.

    This function is the contract. Call it ALWAYS, before a model sees anything.
    """
    if wav is None:
        raise ValueError('No audio received. Please upload a clip.')
    wav = np.asarray(wav)
    if wav.dtype.kind in 'iu':                       # gradio hands back int16
        wav = wav.astype('float32') / np.iinfo(wav.dtype).max
    wav = wav.astype('float32')
    if wav.ndim > 1:                                 # to mono, either layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * MIN_SECONDS:
        raise ValueError(f'Clip too short ({wav.size/sr:.2f} s). Use at least {MIN_SECONDS} s.')
    if wav.size > sr * MAX_SECONDS:
        wav = wav[:int(sr * MAX_SECONDS)]            # truncate, and the caller says so
    if np.abs(wav).max() < 1e-4:
        raise ValueError('That clip is silent. Whisper would invent words for it.')
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype('float32'), target_sr


print('contract loaded. CLAP_SR =', CLAP_SR, '| ASR_SR =', ASR_SR)

### The shape your functions return · **given, and it is the joint with L25**

On Monday you write `analyse_image(...)`. It returns **a dict with these same keys**, and L25's
join is comparing the two `tags` lists. Return a different shape today and Monday does not fit.

```python
{
    'modality': 'audio',            # Monday: 'image'
    'scene'   : 'casa',             # which vocabulary was used
    'tags'    : [('lluvia', 0.91), ('viento', 0.05), ...],   # sorted, highest first
    'text'    : 'what the speech said, or None',
    'summary' : 'one sentence in Spanish',
}
```

## 4 — Your vocabulary · **you write it**

Here is the difference from the L17 classifier. That one had 50 classes somebody else trained.
This one has **whatever you write**, and it changes its answer when you change the wording.

### How that actually works

Your L17 model ends in fifty output neurons. Those fifty were decided before training, and a
fifty-first means training again.

CLAP has **no output neurons at all**. It turns the audio into a vector, turns each of your
phrases into a vector *in the same space*, and answers with whichever is closest.

```
  audio  ──►  ●                     your phrases ──►  ● ● ●
                 nearest one wins
```

So "the classes" are a Python list of strings you pass at call time. Nothing is trained, nothing
is saved, and you can change the whole vocabulary between two calls. This is the same trick CLIP
does for images — which is exactly why Monday's half is short.

Cost does not depend on the list: a few labels or a few dozen, a one-second clip or a
thirty-second one, inference is ~0.15 s either way.

Two measured things worth knowing:

- `'a dog barking'` scores 1.000 where `'dog'` scores 0.983. Descriptive phrases beat bare
  words — CLAP was trained on descriptions, not on labels.
- **CLAP never says "none of these".** A dog, offered only rain / vacuum / birds / engine,
  answers **"birds chirping" at 0.497**. It always returns a ranked list. That is why the spec
  asks for a confidence floor: the "I don't know" is yours to write.

Write **at least two scenes** with **at least four labels each**. Pick sounds you can actually
record or find today — you need them to test.

> **No clips of your own?** There are nine in the shared `inputs/` folder: dog, rain, birds,
> vacuum cleaner, crying baby, ticking clock, a chainsaw, and speech in Spanish and English.
> Run `list_inputs()` to see them. Using your own recordings is better, but not at the cost of
> having nothing to test against.

In [ ]:
# TODO: your vocabulary. Keys are scenes; values are the phrases CLAP compares against.
# Write them in whichever language you like, but measure: CLAP understands English far better.
SCENES = {
    'casa': [
        # 'a dog barking',
        # ...
    ],
    # 'calle': [...],
}

# TODO: below this score, your app refuses to name the sound.
# Do not copy this number from anyone: measure it. Run your clips, see what a hit scores
# and what a miss scores, and put the floor between them.
#
# And know what the floor does NOT do. Measured: a chainsaw, offered a list with no chainsaw
# in it, comes back "a vacuum cleaner" at 0.880 — above any floor you would reasonably set.
# The score is a softmax over YOUR list, so it says "which of these fits best", never "does
# any of these fit". A floor catches the model being unsure. It cannot catch it being wrong.
# If that bothers you, the fixes are a wider vocabulary, an explicit "something else" label,
# or looking at the gap between the top two instead of the top one.
CONF_FLOOR = None

## 5 — The four functions · **you write them**

The signatures and docstrings are given so the joint with L25 works. The bodies are not.

Suggested order: `tag_audio` first, and test it alone on a real clip before going further.
If that one does not give sensible answers, none of the other three can.

In [ ]:
def tag_audio(wav, sr, labels):
    """(wav, sr) + a list of phrases -> [(label, score), ...] sorted, highest first.

    Assumes sr == CLAP_SR. If it is not, that is a bug in the caller: use coerce_audio.
    CLAP returns a list of dicts with 'label' and 'score'.
    """
    assert sr == CLAP_SR, f'{sr} != {CLAP_SR} - coerce first'
    # TODO
    raise NotImplementedError


def transcribe(wav, sr, language=None):
    """(wav, sr) -> text. Whisper wants float32 at 16 kHz."""
    assert sr == ASR_SR, f'{sr} != {ASR_SR} - coerce first'
    # TODO
    raise NotImplementedError


def analyse_audio(wav, sr, scene):
    """(wav, sr) + a scene name -> the dict from section 3b.

    Your logic goes here: tag it, decide whether there is speech, apply the confidence floor,
    and compose `summary` in Spanish. You decide what counts as "there is speech".
    """
    # TODO
    raise NotImplementedError


def run_audio(x, sr=None, scene='casa'):
    """The one function the UI calls. Anything in, and it NEVER raises.

    Returns (summary, report), where `report` is text for the screen — labels, timings,
    whatever helps you debug live. If something fails, return a message a user can
    understand, not a traceback.
    """
    # TODO
    raise NotImplementedError

## 6 — The test cell · **given. It has to pass.**

Do not modify it. If it fails, the problem is in your code, not in the test.

It checks three things: that hostile inputs do not raise, that the contract actually protects
you from the `int16` bug, and that your labels fire on your clips.

In [ ]:
# ---------- A. hostile inputs: none of these may raise ----------
rng   = np.random.default_rng(SEED)
noise = (0.2 * rng.standard_normal(CLAP_SR * 5)).astype('float32')
scene = next(iter(SCENES))

print('--- hostile inputs ---')
cases = [
    ('None',        None,                                          CLAP_SR),
    ('silence',     np.zeros(CLAP_SR * 5, dtype='float32'),        CLAP_SR),
    ('stereo',      np.stack([noise, noise], axis=-1),             CLAP_SR),
    ('int16',       (noise * 32767).astype('int16'),               CLAP_SR),
    ('odd rate',    noise,                                         44100),
    ('0.1 s',       noise[:int(CLAP_SR * 0.1)],                    CLAP_SR),
    ('30 minutes',  np.tile(noise, 360),                           CLAP_SR),
]
fails = 0
for tag, bad, sr in cases:
    try:
        _s, msg = run_audio(bad, sr=sr, scene=scene)
        first = (msg or _s or '').splitlines()[0][:60]
        print(f'  {tag:12s} -> {first}')
    except Exception as e:
        print(f'  {tag:12s} -> RAISED {type(e).__name__}: {e}')
        fails += 1

# ---------- B. the contract is worth something ----------
print('\n--- int16, with and without the contract ---')
i16 = (noise * 32767).astype('int16')
raw  = tag_audio(i16.astype('float32'), CLAP_SR, SCENES[scene])      # unnormalised: wrong
good = tag_audio(*coerce_audio(i16, CLAP_SR, CLAP_SR), SCENES[scene])
print(f'  without coerce_audio : {raw[0][0]:24s} {raw[0][1]:.3f}')
print(f'  with coerce_audio    : {good[0][0]:24s} {good[0][1]:.3f}')
print('  (on white noise the two may agree; on a real clip they almost never do)')

# ---------- C. your clips against your vocabulary ----------
print('\n--- your clips ---')
clips = [f for f in list_inputs() if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac'))]
if not clips:
    print('  (no clips in the folder - run upload_inputs())')
for name in clips[:4]:
    wav, sr = load_audio(name)
    summary, report = run_audio(wav, sr=sr, scene=scene)
    print(f'  {name}: {summary}')

vram('peak')
print(f"\npeak {torch.cuda.max_memory_allocated()/1e9:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print('\nhostile inputs that raised:', fails, '(must be 0)')

## 7 — The app · **you write it**

Everything you need from Gradio you already used in L11 and L12: `Blocks`, `Row`, `Column`,
`Button`, `Textbox`, `Dropdown`, `Markdown`, `.click()`, `.queue()`, `.launch()`.

**The only new thing is `gr.Audio`**, and it carries the trap you saw in the demo:

```python
gr.Audio(type='numpy')   # the callback receives (sr, wav). Rate FIRST.
```

`librosa.load` returns `(wav, sr)`. Gradio returns `(sr, wav)`. Unpack it backwards and nothing
raises — you resample to the length of the array and listen to noise.

Minimum interface requirements:

- `gr.Audio(type='numpy')` as input
- a `gr.Dropdown` to pick the scene, fed from `SCENES`
- a `gr.Textbox` with the report
- a button, and `.queue()` before `.launch(share=True)`

In [ ]:
import gradio as gr
print('gradio', gr.__version__)

# TODO: your app.
#   def on_click(audio, scene):
#       sr, wav = audio            # <- rate FIRST
#       ...

In [ ]:
# Always close before relaunching, or you leak ports.
# demo.close()

---

## If you get stuck

**The fastest way to understand the vocabulary** is to stop reading and run one clip twice,
against two different lists:

```python
wav, sr = coerce_audio(*load_audio('dog.wav'), CLAP_SR)
print(tag_audio(wav, sr, ['a dog barking', 'rain falling']))
print(tag_audio(wav, sr, ['a wolf howling', 'a person laughing']))
```

Same audio, different answers, nothing retrained. Thirty seconds, and the idea lands harder
than any paragraph above.

**If an assistant is writing your CLAP code**, check it against section 2 before you run it.
CLAP is recent and less common than Whisper, and assistants confidently invent APIs for it —
a `CLAPModel.classify()` that does not exist, a `sampling_rate=` argument the pipeline rejects.
The docstrings in this notebook are the ground truth; a fluent answer is not.

**Work out which half is broken** before changing anything. `tag_audio` on a known clip tells
you whether the model or your composition is at fault, and those need different fixes.

---

## Before you leave

- [ ] The test cell runs and reports **0** hostile inputs that raised
- [ ] `analyse_audio` returns the dict with all five keys from section 3b
- [ ] The app opens, accepts a clip, and answers something sensible
- [ ] `CONF_FLOOR` holds a number you **measured**, not one you copied
- [ ] The notebook is saved to Drive
- [ ] **Restart the runtime and run everything again.** A notebook that only runs in the
      current state of memory is not a deliverable, it is a memory.

## Monday (L25)

You get the other half of the skeleton: `analyse_image`, against **the same `SCENES`**. The join
is comparing the `tags` from both halves and saying whether they agree. Then the showcase.

If you did not finish today, Monday starts with finishing. Whatever does not fit goes home as
homework, with no penalty — but everyone shows something.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L24*